# Delta System — Kaggle Training
**Goal**: Train on 8000 NewsEdits pairs, evaluate on 1000 held-out unseen pairs.

**Pass criteria (held-out only):**
- `DELTA_PPL > 2` — delta helps reconstruct B on unseen pairs
- `SPECIFICITY > 2` — correct delta beats wrong delta on unseen pairs

Read `delta_system/SYSTEM.md` for full architecture and context.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install transformers scikit-learn datasets -q

In [ ]:
# ── Cell 2: Setup paths (Kaggle GitHub integration puts repo at /kaggle/working) ─
import os, sys
from pathlib import Path

# Find delta_system — works whether repo is cloned or linked via GitHub
candidates = [
    Path('/kaggle/working/multihop-retrieval/delta_system'),
    Path('/kaggle/working/delta_system'),
    Path('delta_system'),
]
DELTA_DIR = next((p for p in candidates if p.exists()), None)
if DELTA_DIR is None:
    print('Cloning repo...')
    os.system('git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git /kaggle/working/multihop-retrieval')
    DELTA_DIR = Path('/kaggle/working/multihop-retrieval/delta_system')

sys.path.insert(0, str(DELTA_DIR))
print(f'delta_system path: {DELTA_DIR}')
print(f'Files: {[f.name for f in DELTA_DIR.glob("*.py")]}')

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
import torch

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN   = 8000
N_EVAL    = 1000
STEPS     = 2000
BS        = 16
LR        = 1e-4
LAM_S     = 1.0
LAM_SPEC  = 1.0
MARGIN    = 2.0
LOG_EVERY = 200
MAX_LEN   = 128

print(f'Device : {DEVICE}')
print(f'Config : {N_TRAIN} train / {N_EVAL} held-out | {STEPS} steps | bs={BS}')

In [ ]:
# ── Cell 4: Load NewsEdits data ───────────────────────────────────────────────
# NewsEdits (NAACL 2022): Wikipedia article edit history
# A = article before edit, B = article after edit
# We keep only additions where after_text is meaningfully longer

def load_newsedits(n_train=8000, n_eval=1000):
    from datasets import load_dataset
    total = n_train + n_eval
    pairs = []

    # Try several known dataset identifiers for NewsEdits
    dataset_ids = [
        ('wiki_edits', 'train'),
        ('newsedits', 'train'),
        ('RealTimeData/wikiedits', 'train'),
    ]

    ds = None
    for ds_id, split in dataset_ids:
        try:
            print(f'Trying: {ds_id}...')
            ds = load_dataset(ds_id, split=split, streaming=True, trust_remote_code=True)
            print(f'Loaded: {ds_id}')
            break
        except Exception as e:
            print(f'  Failed: {e}')

    if ds is None:
        raise RuntimeError('Could not load any NewsEdits dataset. See fallback cell below.')

    # Inspect first example to find column names
    first = next(iter(ds))
    print(f'Columns: {list(first.keys())}')

    # Reset and collect pairs
    ds = load_dataset(ds_id, split=split, streaming=True, trust_remote_code=True)
    for ex in ds:
        # Try common column name variants
        before = (ex.get('before_sentence') or ex.get('before') or
                  ex.get('old_text') or ex.get('src') or '').strip()
        after  = (ex.get('after_sentence') or ex.get('after') or
                  ex.get('new_text') or ex.get('tgt') or '').strip()

        if not before or not after or len(after) <= len(before):
            continue
        novel = after[len(before):].strip()
        if len(novel) < 20 or len(before) < 20:
            continue

        pairs.append({'A': before, 'B': after, 'novel': novel})
        if len(pairs) >= total:
            break

        if len(pairs) % 1000 == 0:
            print(f'  Collected {len(pairs)}/{total}...')

    print(f'Total pairs: {len(pairs)}')
    return pairs[:n_train], pairs[n_train:total]


train_pairs, eval_pairs = load_newsedits(N_TRAIN, N_EVAL)
print(f'Train: {len(train_pairs)} | Eval (held-out): {len(eval_pairs)}')
print(f'Example A: {train_pairs[0]["A"][:100]}')
print(f'Example novel: {train_pairs[0]["novel"][:100]}')

In [ ]:
# ── Cell 4b: FALLBACK if NewsEdits fails — use MuSiQue uploaded as Kaggle dataset
# Upload musique_ans_v1.0_train.jsonl as a Kaggle dataset first
# Then uncomment and run this cell instead of Cell 4

# import json
# MUSIQUE_PATH = Path('/kaggle/input/musique-dataset/musique_ans_v1.0_train.jsonl')
# pairs = []
# with open(MUSIQUE_PATH) as fh:
#     for line in fh:
#         ex = json.loads(line)
#         decomp = ex.get('question_decomposition', [])
#         if len(decomp) != 2: continue
#         paras = {p['idx']: p['paragraph_text'].strip() for p in ex['paragraphs']}
#         idx1  = decomp[0].get('paragraph_support_idx')
#         idx2  = decomp[1].get('paragraph_support_idx')
#         if idx1 is None or idx2 is None: continue
#         A, novel = paras.get(idx1,'').strip(), paras.get(idx2,'').strip()
#         if A and novel:
#             pairs.append({'A': A, 'B': A+' '+novel, 'novel': novel})
#         if len(pairs) >= N_TRAIN + N_EVAL: break
# train_pairs = pairs[:N_TRAIN]
# eval_pairs  = pairs[N_TRAIN:N_TRAIN+N_EVAL]
# print(f'MuSiQue fallback: {len(train_pairs)} train / {len(eval_pairs)} eval')

In [ ]:
# ── Cell 5: Load model + tokenizer ───────────────────────────────────────────
from model import DeltaSystem
from transformers import BertTokenizerFast

tok   = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = DeltaSystem().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params/1e6:.1f}M')
print(f'BERT frozen: {not model.bert.parameters().__next__().requires_grad}')

In [ ]:
# ── Cell 6: Training ─────────────────────────────────────────────────────────
import math
from torch.utils.data import DataLoader, Dataset
from losses import recon_loss, sparsity_loss, specificity_loss

class PairDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i): return self.pairs[i]['A'], self.pairs[i]['B']

def make_collate(tok):
    def collate(batch):
        A_texts = [x[0] for x in batch]
        B_texts = [x[1] for x in batch]
        eA = tok(A_texts, max_length=MAX_LEN, truncation=True, padding='max_length', return_tensors='pt')
        eB = tok(B_texts, max_length=MAX_LEN, truncation=True, padding='max_length', return_tensors='pt')
        return eA['input_ids'], eA['attention_mask'], eB['input_ids'], eB['attention_mask']
    return collate

dl  = DataLoader(PairDS(train_pairs), batch_size=BS, shuffle=True,
                 collate_fn=make_collate(tok), num_workers=2, pin_memory=True)
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

model.train()
step = 0
while step < STEPS:
    for batch in dl:
        if step >= STEPS: break
        A_ids, A_mask, B_ids, B_mask = [t.to(DEVICE) for t in batch]
        b = A_ids.size(0)

        logits, delta, delta_0, H_A, alpha = model(A_ids, A_mask, B_ids, B_mask)
        L_r    = recon_loss(logits, B_ids, B_mask)
        L_s    = sparsity_loss(delta, B_mask)
        L_spec = torch.tensor(0.0, device=DEVICE)
        if b > 1:
            idx_shift    = list(range(1, b)) + [0]
            logits_wrong = model.reconstruct(
                H_A, A_mask, delta[idx_shift], delta_0[idx_shift], B_ids, B_mask)
            L_spec = specificity_loss(logits, logits_wrong, B_ids, B_mask, margin=MARGIN)

        loss = L_r + LAM_S * L_s + LAM_SPEC * L_spec
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        step += 1

        if step % LOG_EVERY == 0 or step == 1:
            ppl = math.exp(min(L_r.item(), 20))
            print(f'  step {step:4d}/{STEPS} | ppl={ppl:.1f} | L_spec={L_spec.item():.4f}')

print('Training complete.')

In [ ]:
# ── Cell 7: Save checkpoint ───────────────────────────────────────────────────
ckpt_dir = Path('/kaggle/working/checkpoints')
ckpt_dir.mkdir(exist_ok=True)
trainable = {k: v for k, v in model.state_dict().items() if not k.startswith('bert.')}
torch.save(trainable, ckpt_dir / 'kaggle_model.pt')
print(f'Saved: {ckpt_dir / "kaggle_model.pt"}')

In [ ]:
# ── Cell 8: HELD-OUT EVALUATION (unseen pairs) ────────────────────────────────
from eval import evaluate

print('=' * 62)
print(f'HELD-OUT EVAL — {len(eval_pairs)} pairs never seen during training')
print('=' * 62)
held_out_results = evaluate(model, eval_pairs, tok)

In [ ]:
# ── Cell 9: IN-SAMPLE CHECK (sanity — should be much better than held-out) ────
print('=' * 62)
print('IN-SAMPLE CHECK — 200 training pairs (sanity check)')
print('=' * 62)
train_results = evaluate(model, train_pairs[:200], tok)

print('\n' + '=' * 62)
print('SUMMARY')
print('=' * 62)
print(f'  In-sample  DELTA_PPL : {train_results["delta_ppl"]:+.2f}')
print(f'  Held-out   DELTA_PPL : {held_out_results["delta_ppl"]:+.2f}  <- KEY NUMBER')
print(f'  In-sample  SPEC      : {train_results["specificity"]:+.2f}')
print(f'  Held-out   SPEC      : {held_out_results["specificity"]:+.2f}')
print()
if held_out_results['pass']:
    print('  ✓ HELD-OUT PASS — system generalizes. Ready to scale further.')
else:
    print('  ✗ HELD-OUT FAIL — still underfitting. Try more steps or more data.')
print('=' * 62)